#####Importing the necessities

In [0]:
import pyspark.sql.functions as F 
from pyspark.sql.types import FloatType,DoubleType, IntegerType, StringType, StructType, StructField
sil_brands = spark.table("ecommerce.bronze.brz_brands")
sil_category = spark.table("ecommerce.bronze.brz_category")
sil_customers = spark.table("ecommerce.bronze.brz_customers")
sil_product = spark.table("ecommerce.bronze.brz_products")
sil_date = spark.table("ecommerce.bronze.brz_date")

catalogue_name = 'ecommerce'

##### Loaded the data into the Bronze now cleaning and injesting to the silver part 

In [0]:
import pyspark.sql.functions as F

##### Cleangin the spaces in the table **`Brands`** table and inserting in the silver model


In [0]:
from pyspark.sql.functions import trim, col

cols = ['brand_code', 'brand_name', 'category_code']

for c in cols:
    sil_brands = sil_brands.withColumn(c, trim(col(c)))

In [0]:
sil_brands = sil_brands\
    .withColumn("brand_code",F.regexp_replace(F.col('brand_code'),r'[^A-za-z0-9]',''))

In [0]:
# display(sil_brands.select("category_code").distinct())    

In [0]:
from pyspark.sql.functions import col, when

anomalies = {
    'BOOKS': 'BKS',
    'GROCERY': 'GRCY',
    'TOYS': 'TOY'
}

sil_brands = sil_brands.replace(anomalies, subset=['category_code'])

In [0]:
sil_brands.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalogue_name}.silver.sil_brands')


In [0]:
# display(sil_brands.select("category_code").distinct())

##### Cleaning the data in the table **`Category`** table and inserting in the silver 


In [0]:
display(sil_category.limit(20))


In [0]:
sil_category = sil_category.withColumn('category_code',F.upper(F.col("category_code")))

##### Everything looks fine in `Category` so directly inserting into the **silver**

In [0]:
sil_category.write.format('delta')\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable(f'{catalogue_name}.silver.sil_category')

##### Working on the **`Products`** table and inserting to the silver

In [0]:
sil_product = sil_product.withColumn('category_code',F.upper(F.col("category_code")))

sil_product = sil_product.withColumn(
    'weight_grams',
    F.round(F.regexp_replace(F.col('weight_grams'), r'[^0-9.]', '').cast(DoubleType()), 2))

sil_product = sil_product.withColumn(
    'length_cm',
    F.round(F.regexp_replace(F.col('length_cm'), ',', '.').cast(DoubleType()), 2))

sil_product = sil_product.withColumn(
    'height_cm',
    F.round(F.regexp_replace(F.col('height_cm'), ',', '.').cast(DoubleType()), 2))

sil_product = sil_product.withColumn(
    'width_cm',
    F.round(F.regexp_replace(F.col('width_cm'), ',', '.').cast(DoubleType()), 2)
)

sil_product = sil_product.withColumn('brand_code',F.upper(F.col("brand_code")))
# Can also be done by this way
cols = ['category_code','brand_code']
for c in cols:
    sil_product = sil_product.withColumn(c,F.upper(col(c)))


sil_product = sil_product.withColumn('rating_count',F.abs(F.col("rating_count")))

sil_product = sil_product.withColumn(
    "material",
    F.when(F.col("material") == 'Coton', "Cotton")
     .when(F.col("material") == 'Ruber', "Rubber")
     .when(F.col("material") == 'Alumium', 'Aluminium')
     .otherwise(F.col("material"))
)



In [0]:
sil_product.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalogue_name}.silver.sil_product')

##### Cleaning the `Customers` table

In [0]:

row_count =  sil_customers.dropna(subset='customer_id')

sil_customers = sil_customers.dropna(subset='customer_id')
sil_customers = sil_customers.fillna('Not Available',subset = ["phone"])
display(sil_customers.filter(F.col("phone")=='Not Available'))


In [0]:
sil_customers.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{catalogue_name}.silver.sil_customers')


##### Cleaning the `Date` table

In [0]:

sil_date = sil_date.withColumn("week_of_year",F.abs(F.col("week_of_year")))
sil_date = sil_date.withColumn('day_name',F.initcap(F.col('day_name')))
from pyspark.sql.functions import to_date

sil_date = sil_date.withColumn('date', F.to_date(F.col('date'), 'dd-MM-yyyy'))

sil_date=sil_date.dropDuplicates(subset = ['date'])

sil_date = sil_date.withColumn('quarter',F.concat_ws(
    "-",F.concat(F.lit("Q"),F.col("quarter"),F.lit("-"),F.col("year"))
))

sil_date = sil_date.withColumn(
    "week_of_year",
    F.concat_ws(
        "-",
        F.concat(F.lit("Week"), F.col("week_of_year")),
        F.col("year")
    )
)

display(sil_date)   # ← add this here to see the real final state

In [0]:
# sil_date.write.format('delta')\
#     .mode('overwrite')\
#     .option('mergeSchema', True)\
#     .saveAsTable(f'{catalogue_name}.silver.sil_date')